# 08 — KNN + Decision Tree
Two non-linear models with bias-variance analysis and feature importance.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, f1_score
import matplotlib.pyplot as plt
from src.preprocess import build_preprocessor
from src.config import NUMERICAL_FEATURES, CATEGORICAL_FEATURES, RANDOM_STATE

In [ ]:
X_train = pd.read_csv('data/processed/X_train.csv')
X_test = pd.read_csv('data/processed/X_test.csv')
y_train = pd.read_csv('data/processed/y_train.csv').values.ravel()
y_test = pd.read_csv('data/processed/y_test.csv').values.ravel()

## 1. KNN — with Sampling
KNN is O(n*d) at prediction time. We sample 30K training rows for speed.

In [ ]:
SAMPLE_SIZE = 30000
np.random.seed(RANDOM_STATE)
# CORRECT: integer indexing — works with numpy y_train
sample_idx = np.random.choice(len(X_train), size=min(SAMPLE_SIZE, len(X_train)), replace=False)
X_train_knn = X_train.iloc[sample_idx]
y_train_knn = y_train[sample_idx]  # numpy array — integer index works

print(f"KNN training set: {X_train_knn.shape} (sampled from {X_train.shape[0]:,})")
print(f"Class dist in sample: {np.bincount(y_train_knn)}")

### KNN with MinMaxScaler (better for distance-based models)

In [ ]:
# Build MinMaxScaler preprocessor for KNN
knn_num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler())
])
knn_cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
knn_preprocessor = ColumnTransformer(transformers=[
    ('num', knn_num_pipeline, NUMERICAL_FEATURES),
    ('cat', knn_cat_pipeline, CATEGORICAL_FEATURES)
])

# GridSearchCV over k values
knn_pipeline = Pipeline(steps=[
    ('preprocessor', knn_preprocessor),
    ('classifier', KNeighborsClassifier())
])

param_grid = {'classifier__n_neighbors': [3, 5, 7, 11, 15]}
grid_knn = GridSearchCV(knn_pipeline, param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=1)
grid_knn.fit(X_train_knn, y_train_knn)

print(f"Best k: {grid_knn.best_params_}")
print(f"Best CV F1: {grid_knn.best_score_:.4f}")

y_pred_knn = grid_knn.predict(X_test)
print(f"\nTest F1 (class 1): {f1_score(y_test, y_pred_knn, pos_label=1):.4f}")
print(classification_report(y_test, y_pred_knn, target_names=['No Default', 'Default']))

## 2. Decision Tree — Bias-Variance Analysis

In [ ]:
# Train trees with different max_depth values
depths = [3, 5, 7, 10, None]
train_f1s = []
test_f1s = []

for depth in depths:
    dt = Pipeline(steps=[
        ('preprocessor', build_preprocessor()),
        ('classifier', DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE))
    ])
    dt.fit(X_train, y_train)
    train_f1s.append(f1_score(y_train, dt.predict(X_train), pos_label=1))
    test_f1s.append(f1_score(y_test, dt.predict(X_test), pos_label=1))
    print(f'depth={str(depth):>4s} | train F1={train_f1s[-1]:.4f} | test F1={test_f1s[-1]:.4f}')

### Bias-Variance Visualization

In [ ]:
depth_labels = [str(d) if d else 'None' for d in depths]
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(depth_labels, train_f1s, 'o-', label='Train F1', linewidth=2)
ax.plot(depth_labels, test_f1s, 's-', label='Test F1', linewidth=2)
ax.set_xlabel('max_depth')
ax.set_ylabel('F1 Score (class 1)')
ax.set_title('Decision Tree: Bias vs Variance', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.savefig('docs/eda_plots/bias_variance_dt.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Feature Importance (from best Decision Tree)

In [ ]:
best_depth = depths[np.argmax(test_f1s)]
dt_best = Pipeline(steps=[
    ('preprocessor', build_preprocessor()),
    ('classifier', DecisionTreeClassifier(max_depth=best_depth, random_state=RANDOM_STATE))
])
dt_best.fit(X_train, y_train)

# Get feature names after encoding
feature_names = (NUMERICAL_FEATURES + 
    list(dt_best.named_steps['preprocessor']
         .named_transformers_['cat']
         .named_steps['encoder']
         .get_feature_names_out(CATEGORICAL_FEATURES)))

importances = dt_best.named_steps['classifier'].feature_importances_
feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
feat_imp.head(15).plot(kind='barh', ax=ax, color='#e74c3c', edgecolor='black')
ax.set_title('Top 15 Feature Importances — Decision Tree', fontsize=14, fontweight='bold')
ax.invert_yaxis()
plt.savefig('docs/eda_plots/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()